# Open Models in Jupyter

Most of the time you use AI through a website or an app. The model lives somewhere else and you never see it.

Here the model runs right in this notebook. You load it, talk to it, and look at exactly what it sees.

## Part 1: Run the Model

### Find the models

Your instructor already downloaded them. Let's see what we have.

In [ ]:
from pathlib import Path

folder = Path("/home/jovyan/shared/open-models")   # on the hub
if not folder.exists():
    folder = Path("models")                        # on your laptop

for f in sorted(folder.glob("*.gguf")):
    print(f"{f.name:28} {f.stat().st_size / 1e9:.1f} GB")

Each `.gguf` file is a whole model. One file. The number in the name is how many parameters it has, in billions.

### Load one

Start with the smallest.

In [ ]:
import shutil
from llama_cpp import Llama

model = Llama(
    model_path=str(folder / "Qwen3.5-0.8B-Q4_K_M.gguf"),
    n_ctx=4096,                                            # how much text it can look at once
    n_gpu_layers=-1 if shutil.which("nvidia-smi") else 0,  # use the GPU if there is one
    verbose=False,
)

### Ask it something

In [ ]:
messages = [
    {"role": "user", "content": "What is a language model? One sentence."},
]

response = model.create_chat_completion(messages=messages, max_tokens=200)
response

That is everything the model sent back. It is just a Python dictionary. The answer is in here:

In [ ]:
print(response["choices"][0]["message"]["content"])

Look at `usage` in the dictionary too. That is how many tokens went in and how many came out.

### What did the model actually see?

We gave it a list. The model does not read lists. Your messages get turned into one long string with special markers first. The **chat template** builds that string, and it comes inside the model file.

In [ ]:
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

template = Jinja2ChatFormatter(
    template=model.metadata["tokenizer.chat_template"],
    eos_token="",
    bos_token="",
)

prompt = template(messages=messages).prompt
print(prompt)

- `<|im_start|>` and `<|im_end|>` mark where each message starts and stops.
- The empty `<think>` block tells this model to skip thinking out loud and just answer.
- The string stops right where the assistant should start talking. The model's whole job is to keep going.

### Then it becomes numbers

In [ ]:
tokens = model.tokenize(prompt.encode(), add_bos=False, special=True)
print(len(tokens), "tokens")
print(tokens)

In [ ]:
[model.detokenize([t], special=True).decode(errors="replace") for t in tokens]

Every piece of text is a number. The model only ever sees the numbers, and it answers with numbers that get turned back into text.

That token count matches `prompt_tokens` in the response above.

### Try it

1. Change the question and run it again.
2. Set `max_tokens=10`. What happens to the answer? What does `finish_reason` say now?
3. Load a bigger model. Swap in `Qwen3.5-2B-Q4_K_M.gguf` and ask the same question. What changed?